# 基底変換とTT-rank不変性: $U \otimes I$ の小規模検証

$$
X^{\langle 2 \rangle} = (U \otimes I_{n_2}) B^{\langle 2 \rangle}
$$

と

$$
\operatorname{rank}\left(X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(B^{\langle 2 \rangle}\right)
$$

を小さい3階テンソルで確認する。

## このNotebookで何を確認するか

01では、元テンソル $X$ の各cut unfolding rankと、逐次TT-SVDで得られたbond dimensionが一致することを確認した。しかしTT-SVDでは、第1SVDの後は元の $X$ を直接使わず、$B = \Sigma V^\top$ というremainderを次のSVDへ渡している。

そこでこのNotebookでは、**なぜ途中のremainder $B$ を調べても元のテンソルの次のcut rankが分かるのか**を確認する。第2切断で

$$
X^{\langle 2 \rangle} = (U \otimes I_{n_2}) B^{\langle 2 \rangle}
$$

となり、$U$ の列直交性から $U \otimes I$ はrankを変えない。そのため、打ち切りなしでは

$$
\operatorname{rank}\left(X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(B^{\langle 2 \rangle}\right)
$$

が成り立つことを数値的に検証する。最後にtruncationすると表現対象自体が $X \to \hat X$ に変わるため、元の $X$ と $\hat X$ のrankは一致しない場合があることも確認する。

## ゴール
- $U$ の列直交性を確認する
- $U \otimes I$ のshapeを確認する
- 第2切断で等式とrank不変性を確認する
- truncation後は $\hat X$ と $\hat B$ の間でrank不変性が残ることを確認する
- 元の $X$ と $\hat X$ のrankは一致しない場合があることを確認する

> gauge freedomはまだ扱わない。


## 1. 第1切断とSVD

**このセルの目的:** TT-SVDの最初の1段階だけを取り出し、後の検証に使う $U$ と remainder $B = \Sigma V^\top$ を用意する。

$X \in \mathbb{R}^{2 \times 3 \times 4}$ を第1切断 $i_1 \mid i_2 i_3$ でSVDする。ここで作る $U$ は第1coreに対応し、$B$ はTT-SVDで次の段階へ渡される「未分解の残り」に対応する。

このNotebookでは、後で **元の $X$ の第2cut** と **remainder $B$ のcut** を比較したい。その準備として、まず両者を結ぶ $U$ と $B$ を作る。

打ち切りなしでは

$$
r_1 = \operatorname{rank}\left(X^{\langle 1 \rangle}\right)
$$

とする。

> `torch.linalg.matrix_rank` は数値rankを返す。


In [2]:
import torch

from nn_compression.compression import truncated_svd

torch.set_default_dtype(torch.float64)
torch.manual_seed(3)

X = torch.randn(2, 3, 4)
n1, n2, n3 = X.shape

# 第1切断 i1 | i2 i3
X1 = X.reshape(n1, n2 * n3)
r1 = int(torch.linalg.matrix_rank(X1).item())
U, S, Vh = truncated_svd(X1, r1)
B = torch.diag(S) @ Vh

print(f"X.shape = {tuple(X.shape)}")
print(f"X1.shape = {tuple(X1.shape)}")
print(f"r1 = {r1}")
print(f"U.shape = {tuple(U.shape)}")
print(f"B.shape = {tuple(B.shape)}")


X.shape = (2, 3, 4)
X1.shape = (2, 12)
r1 = 2
U.shape = (2, 2)
B.shape = (2, 12)


## 2. $U$ の列直交性

**このセルの目的:** この後で「$U$ を使った変換ではrankが失われない」と言うための前提を数値的に確認する。

SVDで得られる $U$ の列は直交しているので、打ち切りなしで残した列について

$$
U^\top U = I_{r_1}
$$

となる。ここではそのズレを `orth_error` として測り、ほぼ0になることを確認する。

この性質を使って、後の `## 4` で $U \otimes I$ も列直交になることを確認する。


In [3]:
UtU = U.T @ U
I_r1 = torch.eye(r1, dtype=U.dtype, device=U.device)

orth_error = torch.linalg.vector_norm(UtU - I_r1).item()

print(f"U.T @ U =\n{UtU}")
print(f"orth_error = {orth_error:.3e}")


U.T @ U =
tensor([[1.0000, 0.0000],
        [0.0000, 1.0000]])
orth_error = 1.570e-16


## 3. 第2切断

**このセルの目的:** 比較したい2つの行列、元テンソル側の $X^{\langle 2 \rangle}$ と remainder側の $B^{\langle 2 \rangle}$ を作る。

ここではまだrank不変性を確認しない。まず、同じ「第2cut」に対応する形へ $X$ と $B$ をそれぞれ並べ直し、後で両者を直接比較できる状態にする。

$$
X^{\langle 2 \rangle} \in \mathbb{R}^{(n_1 n_2) \times n_3},
\qquad
B^{\langle 2 \rangle} \in \mathbb{R}^{(r_1 n_2) \times n_3}
$$

となるshapeへ変形する。


In [4]:
# 第2切断: i1 i2 | i3
# セクション1の B は (r1, n2*n3) のまま使う。先に上書きしていたらセクション1を再実行する。
print(f"B.shape (expected (r1, n2*n3)=({r1}, {n2 * n3})) = {tuple(B.shape)}")

X_cut2 = X.reshape(n1 * n2, n3)
B_cut2 = B.reshape(r1, n2, n3).reshape(r1 * n2, n3)

print(f"X_cut2.shape = {tuple(X_cut2.shape)}")  # (n1*n2, n3) = (6, 4)
print(f"B_cut2.shape = {tuple(B_cut2.shape)}")  # (r1*n2, n3) = (6, 4)


B.shape (expected (r1, n2*n3)=(2, 12)) = (2, 12)
X_cut2.shape = (6, 4)
B_cut2.shape = (6, 4)


## 4. $U \otimes I$ と列直交性

**このセルの目的:** `## 3` で作った $B^{\langle 2 \rangle}$ を、元の $X^{\langle 2 \rangle}$ 側へ戻す変換 $L_2$ を作り、その変換がrankを潰さないことを確認する。

第1SVDで物理index $i_1$ がbond indexへ変換されたため、第2cutでは $U$ だけでなく、まだ触っていない $i_2$ に対する恒等変換も合わせて

$$
L_2 = U \otimes I_{n_2}
$$

を使う。

$U$ が列直交なら $L_2$ も列直交となり、

$$
L_2^\top L_2 = I
$$

となる。ここで確認したいのは、**$B^{\langle 2 \rangle}$ に $L_2$ を左から掛けてもrankを失うような変換ではない**ということ。


In [ ]:
# L2 = U ⊗ I_{n2}: 第2切断で B_cut2 を X_cut2 側へ戻す左変換
# shape: (n1*n2, r1*n2)
# kronに渡すUを連続メモリ配置にそろえる。
# Uがすでにcontiguousなら、この呼び出しは実質的に追加コピーをしない。
L2 = torch.kron(
    U.contiguous(),
    torch.eye(n2, dtype=U.dtype, device=U.device),
)

# U が列直交なら L2 も列直交: L2^T L2 = I
LtL = L2.T @ L2
I_cols = torch.eye(L2.shape[1], dtype=L2.dtype, device=L2.device)
l2_orth_error = torch.linalg.vector_norm(LtL - I_cols).item()

print(f"L2.shape = {tuple(L2.shape)}")  # (n1*n2, r1*n2) = (6, 6)
print(f"l2_orth_error = {l2_orth_error:.3e}")


L2.shape = (6, 6)
l2_orth_error = 2.719e-16


## 5. 第2切断での関係とrank

**このセルの目的:** ここまで準備した部品を使って、このNotebookの中心となる「remainderのrankを見れば元テンソルの次cut rankが分かる」を直接確認する。

まず

$$
X^{\langle 2 \rangle} = L_2 B^{\langle 2 \rangle}
$$

が数値的に成り立つか確認する。次に、`## 4` で $L_2$ が列直交でrankを失わない変換だと確認したことを使い、

$$
\operatorname{rank}\left(X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(B^{\langle 2 \rangle}\right)
$$

を比較する。

ここが、01で **TT-SVDの途中のremainderを順番にSVDするだけで正しいbond rankが得られた理由**に対応する。


In [18]:
transformed = L2@B_cut2
print(f"L2@B_cut2 =\n{transformed}")
print(f"X_cut2 =\n{X_cut2}")

relation_residual = transformed - X_cut2
relation_error = torch.linalg.matrix_norm(
    relation_residual,
    ord="fro",
).item()

print(f"||X_cut2 - L2 @ B_cut2||_F = {relation_error:.3e}")
print(f"relation_error =\n{relation_error}")
rank_x = int(torch.linalg.matrix_rank(X_cut2).item())
rank_b = int(torch.linalg.matrix_rank(transformed).item())
rank_b_noL = int(torch.linalg.matrix_rank(B_cut2).item())

print(f"rank_x ={rank_x}")
print(f"rank_b ={rank_b}")
print(f"rank_b_noL ={rank_b_noL}")

L2@B_cut2 =
tensor([[-0.2177,  0.1467,  0.6691, -0.5141],
        [-1.5860, -0.7477,  0.6539,  0.9105],
        [ 1.0784,  0.3436,  0.7198,  0.8306],
        [ 0.5087, -1.5266, -1.9011,  0.2482],
        [ 0.0288,  0.2424,  0.7842, -0.9984],
        [ 1.1860,  0.4137,  0.1841, -0.1540]])
X_cut2 =
tensor([[-0.2177,  0.1467,  0.6691, -0.5141],
        [-1.5860, -0.7477,  0.6539,  0.9105],
        [ 1.0784,  0.3436,  0.7198,  0.8306],
        [ 0.5087, -1.5266, -1.9011,  0.2482],
        [ 0.0288,  0.2424,  0.7842, -0.9984],
        [ 1.1860,  0.4137,  0.1841, -0.1540]])
||X_cut2 - L2 @ B_cut2||_F = 7.999e-16
relation_error =
7.999012912691474e-16
rank_x =4
rank_b =4
rank_b_noL =4


## 6. truncationとの違い

**このセルの目的:** `## 5` の「rankが変わらない」という話と、02で行う「rankを削って圧縮する」という話を混同しないようにする。

第1SVDを $\hat r_1 < r_1$ に打ち切ると、元の $X$ をそのまま別の基底で表しているのではなく、情報を捨てた近似テンソル $\hat X$ を表すことになる。

打ち切り後も、残した部分について

$$
\hat X^{\langle 2 \rangle} = \hat L_2 \hat B^{\langle 2 \rangle}
$$

なので、

$$
\operatorname{rank}\left(\hat X^{\langle 2 \rangle}\right) = \operatorname{rank}\left(\hat B^{\langle 2 \rangle}\right)
$$

は成り立つ。

ただし比較対象が元の $X$ から $\hat X$ に変わっているため、一般に

$$
\operatorname{rank}\left(\hat X^{\langle 2 \rangle}\right) \neq \operatorname{rank}\left(X^{\langle 2 \rangle}\right)
$$

となり得る。

ここでは、**基底変換そのものはrankを変えないが、truncationは表現対象を変える**という違いを数値的に確認する。


In [20]:
# 第1SVDを r_hat < r1 に打ち切る（表現対象が X → X_hat に変わる）
r_hat = r1 - 1
U_hat, S_hat, Vh_hat = truncated_svd(X1, r_hat)
B_hat = torch.diag(S_hat) @ Vh_hat

X1_hat = U_hat @ B_hat
X_hat = X1_hat.reshape(n1, n2, n3)

X_hat_cut2 = X_hat.reshape(n1 * n2, n3)
B_hat_cut2 = B_hat.reshape(r_hat, n2, n3).reshape(r_hat * n2, n3)

# SVD 由来の細い U は stride が特殊で kron が落ちることがある。
# flatten → clone → view で行優先の連続配列にしてから Kronecker 積を取る。
U_hat_safe = U_hat.flatten().clone().view(U_hat.shape)
L2_hat = torch.kron(
    U_hat_safe,
    torch.eye(n2, dtype=U_hat.dtype, device=U_hat.device),
)

relation_error_hat = torch.linalg.vector_norm(
    X_hat_cut2 - L2_hat @ B_hat_cut2
).item()

rank_X_hat_cut2 = int(torch.linalg.matrix_rank(X_hat_cut2).item())
rank_B_hat_cut2 = int(torch.linalg.matrix_rank(B_hat_cut2).item())
rank_X_cut2 = int(torch.linalg.matrix_rank(X_cut2).item())

print(f"r_hat = {r_hat}  (original r1 = {r1})")
print(f"relation_error_hat = {relation_error_hat:.3e}")
print(f"rank(X_hat_cut2) = {rank_X_hat_cut2}")
print(f"rank(B_hat_cut2) = {rank_B_hat_cut2}")
print(f"rank(X_cut2)     = {rank_X_cut2}")
print(f"hat ranks match: {rank_X_hat_cut2 == rank_B_hat_cut2}")
print(f"hat vs original: {rank_X_hat_cut2} vs {rank_X_cut2}")


r_hat = 1  (original r1 = 2)
relation_error_hat = 0.000e+00
rank(X_hat_cut2) = 3
rank(B_hat_cut2) = 3
rank(X_cut2)     = 4
hat ranks match: True
hat vs original: 3 vs 4


## 7. 確認

**このセルの目的:** 1〜6で確認したことを、TT-SVDの理解としてまとめる。

- 打ち切りなしTT-SVDでは、途中で $X \to B$ と表現を変えても、列直交な変換を介しているため次cutのrankは失われない
- そのため、逐次SVDで得るbond dimensionと元テンソルのcut unfolding rankが一致する
- truncationすると表現対象が $X \to \hat X$ に変わる
- それでも $\hat X$ と $\hat B$ の間ではrank不変性が成り立つ

ことを確認する。
